# Actividad 2 - Construyendo la Tubería de Datos (tf.data)

**Objetivo:** En el mundo del Big Data, los datos no caben en la memoria RAM. En esta sesión de 2 horas aprenderemos a usar la API `tf.data` para crear un pipeline profesional que lea datos del disco duro, los transforme y los entregue a la tarjeta gráfica en lotes (*batches*) de forma eficiente.

Asegúrate de tener la documentación de TensorFlow a mano.

### Importación de librerías necesarias

In [32]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
from PIL import Image

### Preparación del Entorno (Ejecuta esta celda y no la modifiques)
Esta celda creará un archivo CSV de prueba y algunas imágenes falsas en tu entorno para que tengas datos simulando un caso real. Le cuesta un poco así que ten paciencia.

In [33]:
# 1. Crear un CSV falso
df = pd.DataFrame({'edad': np.random.randint(18, 60, 100), 'salario': np.random.randint(20000, 60000, 100), 'compro': np.random.randint(0, 2, 100)})
df.to_csv('datos_clientes.csv', index=False)

# 2. Crear 5 imágenes falsas en una carpeta
os.makedirs('dataset_imagenes', exist_ok=True)
for i in range(5):
    img = Image.fromarray(np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8))
    img.save(f'dataset_imagenes/img_{i}.jpg')

print("Entorno preparado. Tienes 'datos_clientes.csv' y una carpeta 'dataset_imagenes' listos para usar.")

Entorno preparado. Tienes 'datos_clientes.csv' y una carpeta 'dataset_imagenes' listos para usar.


--- 
### Parte 1: Pipeline básico
Vamos a empezar creando el pipeline más básico posible. Tenemos un array con los números del 1 al 10. Queremos convertirlo en un `Dataset` y agruparlo en lotes de 3.

**Pasos:**
1. Usa `tf.data.Dataset.from_tensor_slices()` para crear el dataset a partir de la lista.
2. Aplica la función `.batch(tamaño)` para agrupar de 3 en 3.
3. Itera sobre el dataset con un bucle `for` e imprime los lotes.

In [34]:
datos_simples = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# 1. Crea el dataset
dataset_simple = tf.data.Dataset.from_tensor_slices(datos_simples)

# 2. Agrupa en lotes de 3
dataset_simple = dataset_simple.batch(3)

# 3. Itera e imprime
print("Lotes generados:")
for lote in dataset_simple:
    print(lote.numpy())

Lotes generados:
[1 2 3]
[4 5 6]
[7 8 9]
[10]


--- 
### Parte 2: Datos Tabulares y la "Trinidad" del Pipeline
Ahora nos enfrentamos a un caso más real. Tienes el archivo `datos_clientes.csv` en tu disco duro. Vamos a cargarlo con Pandas, pasarlo a TensorFlow y aplicar los tres pilares del rendimiento: **Shuffle, Batch y Prefetch**.

**Instrucciones con pistas:**
1. Lee el CSV con Pandas.
2. Convierte el DataFrame entero a un Dataset (igual que en la Parte 1).
3. **Shuffle:** Mezcla los datos usando `.shuffle(buffer_size)`. Como el dataset tiene 100 filas, usa un buffer de 100 para una mezcla perfecta.
4. **Batch:** Agrúpalos en lotes de 10.
5. **Prefetch:** Aplica `.prefetch()`. *Pista: usa `tf.data.AUTOTUNE` dentro del paréntesis para que TF gestione la memoria por ti.*

In [35]:
# 1. Leer con Pandas
df_clientes = pd.read_csv('./datos_clientes.csv')
print(f"Filas en el dataset: {len(df_clientes)}")

# 2. Crear Dataset de TensorFlow desde el DataFrame
dataset_clientes = tf.data.Dataset.from_tensor_slices(df_clientes)

# 3, 4 y 5. Encadenar Shuffle, Batch y Prefetch en una sola línea fluida
# Pista de sintaxis: dataset.shuffle(...).batch(...).prefetch(...)
pipeline_clientes = dataset_clientes.shuffle(100).batch(10).prefetch(tf.data.AUTOTUNE)

# Comprobación: Imprimir solo el primer lote para ver si funciona
for lote in pipeline_clientes.take(1):
    print("\nPrimer lote procesado:\n", lote)

Filas en el dataset: 100

Primer lote procesado:
 tf.Tensor(
[[   30 26151     0]
 [   37 40021     0]
 [   54 51755     0]
 [   39 25242     1]
 [   18 55451     1]
 [   29 21335     1]
 [   35 42014     1]
 [   47 42285     0]
 [   37 42400     1]
 [   37 44403     1]], shape=(10, 3), dtype=int64)


--- 
### Parte 3: El Pipeline de Visión Artificial
Tienes una carpeta llamada `dataset_imagenes` llena de archivos JPG. Las imágenes no se pueden meter crudas a una red neuronal; hay que leer el archivo binario, decodificar el JPEG, redimensionarlo a un tamaño fijo y escalar sus píxeles.

**Procedimiento:**
1. Usa `tf.data.Dataset.list_files('dataset_imagenes/*.jpg')` para crear un dataset que solo contenga las **rutas** de los archivos.
2. Crea una función de Python llamada `procesar_imagen(ruta)` que haga lo siguiente:
   - Leer el archivo del disco. *Investiga: `tf.io.read_file`*
   - Decodificar el JPG a tensores. *Investiga: `tf.image.decode_jpeg` (asegúrate de poner channels=3)*
   - Redimensionar la imagen a 224x224. *Investiga: `tf.image.resize`*
   - Normalizar los píxeles (dividir el tensor entre 255.0 para que vayan de 0 a 1).
3. Usa el método `.map(procesar_imagen)` sobre tu dataset de rutas para aplicar esta función a cada imagen mientras se carga.
4. Aplica `.batch(2)` y `.prefetch(tf.data.AUTOTUNE)` al final.

In [36]:
# 1. Dataset de rutas
rutas_dataset = tf.data.Dataset.list_files('./dataset_imagenes/*.jpg')

# 2. Función de procesamiento (INVESTIGA LOS COMANDOS EXACTOS)
def procesar_imagen(ruta):
    # A. Leer el archivo binario
    archivo = tf.io.read_file(ruta)
    
    # B. Decodificar
    img_tensor = tf.image.decode_jpeg(archivo, channels = 3)
    
    # C. Redimensionar
    img_redimensionada = tf.image.resize(img_tensor, [224, 224])
    
    # D. Normalizar
    img_normalizada = img_redimensionada / 255.0
    
    return img_normalizada

# 3 y 4. Construir el pipeline completo (Map -> Batch -> Prefetch)
pipeline_imagenes = rutas_dataset.map(procesar_imagen).batch(2).prefetch(tf.data.AUTOTUNE)

# Comprobación del experto:
# Extraemos un lote y comprobamos su forma (shape). 
# Si todo ha ido bien, debería imprimir: (2, 224, 224, 3)
for lote_imagenes in pipeline_imagenes.take(1):
    print("Forma del lote de imágenes:", lote_imagenes.shape)
    print("Valor máximo del píxel (debería ser <= 1.0):", tf.reduce_max(lote_imagenes).numpy())

Forma del lote de imágenes: (2, 224, 224, 3)
Valor máximo del píxel (debería ser <= 1.0): 0.99702454
